<a href="https://colab.research.google.com/github/artstudio-mayer/Finanzen/blob/main/Datenaufbereitung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Script für die Finanzübersicht


# Daten laden

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import requests
import plotly.graph_objects as go
import plotly.figure_factory as ff

In [ ]:
url="https://artstudio-mayer.de/finanzen/"
html = requests.get(url).text

In [ ]:
tables = pd.read_html(html)

/tmp/ipython-input-1892305996.py:1: FutureWarning:

Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.



In [ ]:
Datengrundlage_Tablepress = tables[0]

In [ ]:
# Datengrundlage_Tablepress

# Datenbereinigung

In [ ]:
# Datenbereinigung

Datengrundlage_Tablepress["Maße"] = Datengrundlage_Tablepress["Maße"].fillna("")
Datengrundlage_Tablepress["Saldo"] = Datengrundlage_Tablepress["Saldo"].apply(lambda x: f"€{x:,.2f}" if pd.notna(x) and isinstance(x, (int, float)) else x)
Datengrundlage_Tablepress["Einnahme"] = Datengrundlage_Tablepress["Einnahme"].apply(lambda x: f"€{x:,.2f}" if pd.notna(x) and isinstance(x, (int, float)) else x).replace("- €", 0)
Datengrundlage_Tablepress["Ausgabe"] = Datengrundlage_Tablepress["Ausgabe"].apply(lambda x: f"€{x:,.2f}" if pd.notna(x) and isinstance(x, (int, float)) else x)
Datengrundlage_Tablepress["Menge"] = Datengrundlage_Tablepress["Menge"].fillna("0")
Datengrundlage_Tablepress["Datum"] = pd.to_datetime(Datengrundlage_Tablepress["Datum"], errors="coerce")
Datengrundlage_Tablepress["Jahr"] = Datengrundlage_Tablepress["Datum"].dt.year


/tmp/ipython-input-3651252762.py:8: UserWarning:

Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.



In [ ]:
# Bereinigung der Spalte "Saldo"

def bereinige_saldo(wert):
    try:
        bereinigt = wert.replace("€", "").replace(",", ".").replace("–", "").strip()
        return float(bereinigt) if bereinigt else None
    except:
        return None

Datengrundlage_Tablepress['Saldo'] = Datengrundlage_Tablepress['Saldo'].apply(bereinige_saldo).fillna("0")

In [ ]:
# Bereinigung der Spalte "Einnahmen"

def bereinige_einnahme(wert):
    try:
        bereinigt = wert.replace("€", "").replace(",", ".").replace("–", "").strip()
        return float(bereinigt) if bereinigt else None
    except:
        return None

Datengrundlage_Tablepress['Einnahme'] = Datengrundlage_Tablepress['Einnahme'].apply(bereinige_einnahme).fillna("0")

In [ ]:
# Bereinigung der Spalte "Ausgabe"

def bereinige_ausgabe(wert):
    try:
        bereinigt = wert.replace("€", "").replace(",", ".").replace("–", "").strip()
        return float(bereinigt) if bereinigt else None
    except:
        return None

Datengrundlage_Tablepress['Ausgabe'] = Datengrundlage_Tablepress['Ausgabe'].apply(bereinige_ausgabe).fillna("0")

In [ ]:
Datengrundlage_Tablepress

,Kategorie,Posten,Maße,Datum,Saldo,Einnahme,Ausgabe,Menge,Klassifizierung,Notiz,Rechnungen,Jahr
0,Ausgaben,Einkauf Amazon,,2024-02-21,62.0,0,62.0,0,Amazon,Ringlicht mit Stativ,NaN,2024
1,Einnahmen,Unnamed,70cm x 50cm,2024-02-24,71.0,71.0,0,1.0,Freunde/Bekannte,Verkauf an Stefan Uebelacker,NaN,2024
2,Einnahmen,Unnamed,50cm x 40cm,2024-02-27,45.0,45.0,0,1.0,Freunde/Bekannte,Verkauf an Maximilian Mittermeier,NaN,2024
3,Ausgaben,Einkauf Action,,2024-03-21,5.98,0,5.98,0,Action,Bilderrahmen,NaN,2024
4,Ausgaben,Einkauf Jysk,,2024-03-21,49.5,0,49.5,0,Jysk,Bilderrahmen,NaN,2024
...,...,...,...,...,...,...,...,...,...,...,...,...
102,Ausgaben,Einkauf Toom,,2026-01-02,25.96,0,25.96,0.0,Toom,Materialkauf,https://drive.google.com/file/d/1Mm0p3dudqZAZP...,2026
103,Ausgaben,Einkauf Toom,,2026-01-05,67.99,0,67.99,0.0,Toom,Gehrungssäge,https://drive.google.com/file/d/1SKWcIRCqIHjmt...,2026
104,Einnahmen,NaN,,2026-01-01,0,0,0,0,NaN,NaN,NaN,2026
105,Einnahmen,Großer Arber,80cm x 60cm,2026-01-10,90.0,90.0,0,1.0,Website,Verkauf Schiebe (Auftragsarbeit),NaN,2026


In [ ]:
# Datengrundlage_Tablepress.to_excel("output.xlsx", index=False)

# Übersicht

In [ ]:

# ---- 1) Datentypen säubern ----
# Numerik sicherstellen, fehlerhafte Einträge -> 0
for col in ["Menge", "Einnahme", "Ausgabe"]:
    Datengrundlage_Tablepress[col] = pd.to_numeric(Datengrundlage_Tablepress[col], errors="coerce").fillna(0)

# Jahr sicherstellen (aus Datum ableiten, falls Jahr leer ist)
if ("Jahr" not in Datengrundlage_Tablepress.columns) or Datengrundlage_Tablepress["Jahr"].isna().all():
    Datengrundlage_Tablepress["Datum"] = pd.to_datetime(Datengrundlage_Tablepress.get("Datum"), errors="coerce")
    Datengrundlage_Tablepress["Jahr"] = Datengrundlage_Tablepress["Datum"].dt.year

Datengrundlage_Tablepress = Datengrundlage_Tablepress.dropna(subset=["Jahr"]).copy()
Datengrundlage_Tablepress["Jahr"] = pd.to_numeric(Datengrundlage_Tablepress["Jahr"], errors="coerce").astype(int)

# ---- 2) Gruppieren & Summen berechnen ----
agg = (
    Datengrundlage_Tablepress.groupby("Jahr", as_index=False)
      .agg({
          "Menge": "sum",
          "Einnahme": "sum",
          "Ausgabe": "sum",
      })
      .rename(columns={
          "Menge": "_sum_menge",
          "Einnahme": "_sum_einnahme",
          "Ausgabe": "_sum_ausgabe"
      })
)

# ---- 3) Kennzahlen berechnen ----
Übersicht = pd.DataFrame({
    "Jahr": agg["Jahr"],
    "Verkaufte Bilder": agg["_sum_menge"],
    "Umsatz pro Bild": np.where(
        agg["_sum_menge"] > 0,
        agg["_sum_einnahme"] / agg["_sum_menge"],
        0.0
    ),
    "Umsatz": agg["_sum_einnahme"],
    "Ausgaben": agg["_sum_ausgabe"],
    "Gewinn/Verlust": agg["_sum_einnahme"] - agg["_sum_ausgabe"],
}).sort_values("Jahr").reset_index(drop=True)

# ---- 4) Formatierung der Geldbeträge ----
geld_cols = ["Umsatz pro Bild", "Umsatz", "Ausgaben", "Gewinn/Verlust"]

# auf 2 Nachkommastellen runden (technisch)
Übersicht[geld_cols] = Übersicht[geld_cols].round(2)

# deutsche Formatierung als Strings: 1.234,56 € (Tausenderpunkt / Komma als Dezimal)
def format_eur_de(val: float) -> str:
    s = f"{val:,.2f}"         # US-Format: 1,234.56
    s = s.replace(",", "X")   # temporär Komma -> X
    s = s.replace(".", ",")   # Punkt -> Komma
    s = s.replace("X", ".")   # X -> Punkt (Tausender)
    return f"{s} €"

for col in geld_cols:
    Übersicht[col] = Übersicht[col].apply(format_eur_de)

# # verkaufte Bilder als int darstellen (ohne .0)
Übersicht["Verkaufte Bilder"] = Übersicht["Verkaufte Bilder"].astype(int)

# ---- 5) DataFrame -> Matrix (Header + Zeilen) für ff.create_table ----
data_matrix = [Übersicht.columns.tolist()] + Übersicht.values.tolist()

# ---- 6) Plotly-Tabelle erzeugen ----
Übersicht = ff.create_table(data_matrix)
Übersicht.update_layout(
    title="Übersicht",
    width=900,
)
Übersicht.show()


# Test

In [162]:

import pandas as pd
import json

# -----------------------------
# 1) Daten vorbereiten
# -----------------------------
# Beispiel: Dein DataFrame
df = Datengrundlage_Tablepress.copy()

# Nur relevante Spalten
new_df = df[['Jahr', 'Kategorie', 'Klassifizierung', 'Saldo']].copy()

# Saldo numerisch
new_df['Saldo'] = pd.to_numeric(new_df['Saldo'], errors='coerce')

# Ungültige Zeilen entfernen
new_df = new_df.dropna(subset=['Jahr', 'Kategorie', 'Klassifizierung', 'Saldo'])

# Daten als JSON für JavaScript
data_js = json.dumps(new_df.to_dict(orient='records'))

# -----------------------------
# 2) HTML-Template mit eingebettetem JS
# -----------------------------
html_template = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Interaktives Kreisdiagramm</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
</head>
<body>
    <h1>Interaktives Kreisdiagramm</h1>
    <label for="jahrSelect">Jahr:</label>
    <select id="jahrSelect"></select>
    <label for="katSelect">Kategorie:</label>
    <select id="katSelect"></select>
    <div id="chart" style="width:800px;height:600px;"></div>

    <script>
        const data = {data_js};

        // Dropdowns befüllen
        const jahre = [...new Set(data.map(d => d.Jahr))];
        const kategorien = [...new Set(data.map(d => d.Kategorie))];

        const jahrSelect = document.getElementById('jahrSelect');
        const katSelect = document.getElementById('katSelect');

        jahre.forEach(j => {{
            const opt = document.createElement('option');
            opt.value = j;
            opt.textContent = j;
            jahrSelect.appendChild(opt);
        }});

        kategorien.forEach(k => {{
            const opt = document.createElement('option');
            opt.value = k;
            opt.textContent = k;
            katSelect.appendChild(opt);
        }});

        // Standardauswahl
        jahrSelect.value = jahre[0];
        katSelect.value = kategorien[0];

        function updateChart() {{
            const selectedYear = parseInt(jahrSelect.value);
            const selectedCategory = katSelect.value;

            const filtered = data.filter(d => d.Jahr === selectedYear && d.Kategorie === selectedCategory);

            // Aggregation nach Klassifizierung
            const agg = {{}};
            filtered.forEach(d => {{
                agg[d.Klassifizierung] = (agg[d.Klassifizierung] || 0) + d.Saldo;
            }});

            const labels = Object.keys(agg);
            const values = Object.values(agg);

            const trace = {{
                type: 'pie',
                labels: labels,
                values: values,
                hole: 0.35,
                textinfo: 'label+percent',
                hovertemplate: '<b>%{{label}}</b><br>Saldo: %{{value:,.2f}} €<extra></extra>'
            }};

            const layout = {{
                title: `Kreisdiagramm – Jahr ${{selectedYear}}, Kategorie ${{selectedCategory}}`
            }};

            Plotly.newPlot('chart', [trace], layout);
        }}

        // Initiales Diagramm
        updateChart();

        // Event-Listener
        jahrSelect.addEventListener('change', updateChart);
        katSelect.addEventListener('change', updateChart);
    </script>
</body>
</html>
"""

# -----------------------------
# 3) HTML-Datei speichern
# -----------------------------


with open("Interaktives_Kreisdiagramm.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("✅ HTML-Datei erstellt: Interaktives_Kreisdiagramm.html")


✅ HTML-Datei erstellt: Interaktives_Kreisdiagramm.html


# Durchschnittseinnahmen je Maß Tabelle

In [ ]:
# Zahlenrobust konvertieren (de-DE Formate wie "1.234,56" werden unterstützt)
for col in ["Menge", "Einnahme"]:
    Datengrundlage_Tablepress[col] = (
        Datengrundlage_Tablepress[col].astype(str)
    )
    Datengrundlage_Tablepress[col] = pd.to_numeric(Datengrundlage_Tablepress[col], errors="coerce")


# Jahr in Integer
Datengrundlage_Tablepress["Jahr"] = pd.to_numeric(Datengrundlage_Tablepress["Jahr"], errors="coerce").astype("Int64")


# Gruppieren nach Jahr und Maße
grouped = (
    Datengrundlage_Tablepress.groupby(["Jahr", "Maße"], dropna=False)
      .agg(
          Menge_verkauft=("Menge", "sum"),
          Einnahmen=("Einnahme", "sum")
      )
      .reset_index()
)


# Durchschnitt berechnen
grouped["Einnahmen_pro_Maß"] = np.where(
    (grouped["Menge_verkauft"].fillna(0) == 0),
    np.nan,
    grouped["Einnahmen"] / grouped["Menge_verkauft"]
)


# Optional sortieren
grouped = grouped.sort_values(["Jahr", "Maße"]).reset_index(drop=True)


# 1) Menge_verkauft als ganze Zahl anzeigen (ohne ".0")
#    Wir behalten zusätzlich eine numerische Kopie für spätere Berechnungen.
grouped["Menge_verkauft_num"] = grouped["Menge_verkauft"]
# Für die Anzeige:
grouped["Menge_verkauft"] = grouped["Menge_verkauft_num"].fillna(0).astype(int)


# 2) EUR-Format (de-DE) für Einnahmen & Durchschnitt (als Strings für die Tabelle)
def format_eur_de(x):
    if pd.isna(x):
        return ""
    # Format mit Tausendertrennzeichen und Punkt, dann ins deutsche Format umwandeln
    s = f"{x:,.2f}"          # z.B. "12,345.67" (US)
    s = s.replace(",", "X")  # "12X345.67"
    s = s.replace(".", ",")  # "12X345,67"
    s = s.replace("X", ".")  # "12.345,67"
    return s + " €"

grouped["Einnahmen_fmt"] = grouped["Einnahmen"].apply(format_eur_de)
grouped["Durchschnitt_fmt"] = grouped["Einnahmen_pro_Maß"].apply(format_eur_de)


# 3) Anzeige-DataFrame für die Plotly-Tabelle zusammenstellen (nur die gewünschten Spalten)
Maße_Tabelle_Zwischenlösung = grouped[["Jahr", "Maße", "Menge_verkauft", "Einnahmen_fmt", "Durchschnitt_fmt"]].rename(
    columns={
        "Menge_verkauft": "Menge",
        "Einnahmen_fmt": "Einnahmen",
        "Durchschnitt_fmt": "Avg. Einnahmen/Maß"
    }
)


# -------- Plotly Tabelle --------
Maße_Tabelle = ff.create_table(Maße_Tabelle_Zwischenlösung, index=False)
Maße_Tabelle.update_layout(
    title="Übersicht Maße",
    width=900,
)
Maße_Tabelle.show()


# Einnahmen und Ausgaben gegenüberstellen

In [ ]:
# Zahlenfelder sicherstellen
Datengrundlage_Tablepress["Einnahme"] = pd.to_numeric(Datengrundlage_Tablepress["Einnahme"], errors="coerce").fillna(0)
Datengrundlage_Tablepress["Ausgabe"] = pd.to_numeric(Datengrundlage_Tablepress["Ausgabe"], errors="coerce").fillna(0)

In [ ]:
# EuA = Einnahmen und Ausgaben

EuA = Datengrundlage_Tablepress
EuA['Datum'] = pd.to_datetime(Datengrundlage_Tablepress['Datum'])
EuA['Jahr'] = EuA['Datum'].dt.year
EuA['Quartal'] = EuA['Datum'].dt.to_period('Q').astype(str)
EuA['Monat'] = EuA['Datum'].dt.to_period('M').astype(str)

# Aggregation
yearly = EuA.groupby('Jahr')[['Einnahme', 'Ausgabe']].sum().reset_index()
quarterly = EuA.groupby('Quartal')[['Einnahme', 'Ausgabe']].sum().reset_index()
monthly = EuA.groupby('Monat')[['Einnahme', 'Ausgabe']].sum().reset_index()

# Formatierte Labels
quarter_labels = [f"Q{q[-1]} {q[:4][-2:]}" for q in quarterly['Quartal']]
month_labels = [f"{m[5:7]} {m[:4][-2:]}" for m in monthly['Monat']]

# Grunddiagramm
Einnahmen_Ausgaben = go.Figure([
    go.Bar(x=yearly['Jahr'], y=yearly['Einnahme'], name='Einnahme', marker_color='green',
           text=[f"{v:.2f} €" for v in yearly['Einnahme']], textposition='outside'),
    go.Bar(x=yearly['Jahr'], y=yearly['Ausgabe'], name='Ausgabe', marker_color='red',
           text=[f"{v:.2f} €" for v in yearly['Ausgabe']], textposition='outside')
])

Einnahmen_Ausgaben.update_layout(
    title='Einnahmen und Ausgaben',
    xaxis_title='Zeitraum',
    yaxis_title='Betrag in Euro',
    barmode='group',
    xaxis=dict(tickmode='array', tickvals=yearly['Jahr'], ticktext=[str(y) for y in yearly['Jahr']])
)

# Buttons mit Layout-Update
Einnahmen_Ausgaben.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            buttons=[
                dict(label="Jahr",
                     method="update",
                     args=[{"x": [yearly['Jahr'], yearly['Jahr']],
                            "y": [yearly['Einnahme'], yearly['Ausgabe']],
                            "text": [[f"{v:.2f} €" for v in yearly['Einnahme']],
                                     [f"{v:.2f} €" for v in yearly['Ausgabe']]]},
                           {"xaxis": {"tickvals": yearly['Jahr'],
                                      "ticktext": [str(y) for y in yearly['Jahr']]}}]),
                dict(label="Quartal",
                     method="update",
                     args=[{"x": [quarterly['Quartal'], quarterly['Quartal']],
                            "y": [quarterly['Einnahme'], quarterly['Ausgabe']],
                            "text": [[f"{v:.2f} €" for v in quarterly['Einnahme']],
                                     [f"{v:.2f} €" for v in quarterly['Ausgabe']]]},
                           {"xaxis": {"tickvals": quarterly['Quartal'],
                                      "ticktext": quarter_labels}}]),
                dict(label="Monat",
                     method="update",
                     args=[{"x": [monthly['Monat'], monthly['Monat']],
                            "y": [monthly['Einnahme'], monthly['Ausgabe']],
                            "text": [[f"{v:.2f} €" for v in monthly['Einnahme']],
                                     [f"{v:.2f} €" for v in monthly['Ausgabe']]]},
                           {"xaxis": {"tickvals": monthly['Monat'],
                                      "ticktext": month_labels}}])
            ],
            showactive=True,
            x=0.5, xanchor="center",
            y=-0.2, yanchor="top"
        )
    ]
)

Einnahmen_Ausgaben.show()

# Verkäufe nach Maß

In [ ]:
# Zahlenfelder sicherstellen
# Datengrundlage_Tablepress["Einnahme"] = pd.to_numeric(Datengrundlage_Tablepress["Einnahme"], errors="coerce").fillna(0)

In [163]:
## EnM_kum = Einnahmen nach Maß kumuliert
EnM_kum = Datengrundlage_Tablepress

# Menge in numerisch umwandeln
#EnM_kum['Menge'] = pd.to_numeric(EnM_kum['Menge'], errors='coerce')

# Gruppieren nach 'Maße' und Aggregation
#grouped = EnM_kum.groupby('Maße').agg({'Menge': 'sum', 'Einnahme': 'sum'}).reset_index()

# Sortieren nach Einnahme absteigend
#grouped = grouped.sort_values(by='Einnahme', ascending=False)

# Kumulierte prozentuale Einnahme berechnen und runden
#grouped['Einnahme_kumuliert'] = grouped['Einnahme'].cumsum()
#grouped['Einnahme_kumuliert_prozent'] = (100 * grouped['Einnahme_kumuliert'] / grouped['Einnahme'].sum()).round(0)

# X-Achse: Originalwerte aus 'Maße'
#x_labels = grouped['Maße'].tolist()

# Pareto-Diagramm erstellen
#Einnahmen_Maß_kum = go.Figure()

# Balkendiagramm für Menge mit Datenbeschriftung
#Einnahmen_Maß_kum.add_trace(go.Bar(
#    x=x_labels,
#    y=grouped['Menge'],
#    name='Menge',
#    marker_color='blue',
#    yaxis='y1',
#    text=[str(int(m)) for m in grouped['Menge']],
#    textposition='outside'
#))

# Liniendiagramm für kumulierte prozentuale Einnahme (gerundet)
#Einnahmen_Maß_kum.add_trace(go.Scatter(
#    x=x_labels,
#    y=grouped['Einnahme_kumuliert_prozent'],
#    name='Kumulierte Einnahme (%)',
#    marker_color='orange',
#    yaxis='y2',
#    mode='lines+markers+text',
#    text=[f"{int(p)}%" for p in grouped['Einnahme_kumuliert_prozent']],
#    textposition='top center'
#))

# Layout mit zwei Y-Achsen
#Einnahmen_Maß_kum.update_layout(
#    title='Prozentuale Einnahmen pro Maß [kumuliert]',
#    xaxis=dict(title='Maße'),
#    yaxis=dict(title='Menge', side='left'),
#    yaxis2=dict(title='Kumulierte Einnahme (%)', overlaying='y', side='right', range=[0, 110]),
#    legend=dict(x=0.5, xanchor='center', y=-0.2, orientation='h')
#)

#Einnahmen_Maß_kum.show()

In [164]:
# EnM = Einnahmen nach Maß
#EnM = Datengrundlage_Tablepress

# Menge in numerisch umwandeln
#EnM['Menge'] = pd.to_numeric(EnM['Menge'], errors='coerce')

# Gruppieren nach 'Maße' und Aggregation
#grouped = EnM.groupby('Maße').agg({'Menge': 'sum', 'Einnahme': 'sum'}).reset_index()

# Sortieren nach Einnahme absteigend
#grouped = grouped.sort_values(by='Einnahme', ascending=False)

# Kumulierte prozentuale Einnahme berechnen und runden
#grouped['Einnahme_kumuliert'] = grouped['Einnahme']
#grouped['Einnahme_kumuliert_prozent'] = (100 * grouped['Einnahme_kumuliert'] / grouped['Einnahme'].sum()).round(0)

# X-Achse: Originalwerte aus 'Maße'
#x_labels = grouped['Maße'].tolist()

# Pareto-Diagramm erstellen
#Einnahmen_Maß = go.Figure()

# Balkendiagramm für Menge mit Datenbeschriftung
#Einnahmen_Maß.add_trace(go.Bar(
#    x=x_labels,
#    y=grouped['Menge'],
#    name='Menge',
#    marker_color='blue',
#    yaxis='y1',
#    text=[str(int(m)) for m in grouped['Menge']],
#    textposition='outside'
#))

# Liniendiagramm für kumulierte prozentuale Einnahme (gerundet)
#Einnahmen_Maß.add_trace(go.Scatter(
#    x=x_labels,
#    y=grouped['Einnahme_kumuliert_prozent'],
#    name='Kumulierte Einnahme (%)',
#    marker_color='orange',
#    yaxis='y2',
#    mode='lines+markers+text',
#    text=[f"{int(p)}%" for p in grouped['Einnahme_kumuliert_prozent']],
#    textposition='top center'
#))

# Layout mit zwei Y-Achsen
#Einnahmen_Maß.update_layout(
#    title='Prozentuale Einnahmen pro Maß',
#    xaxis=dict(title='Maße'),
#    yaxis=dict(title='Menge', side='left'),
#    yaxis2=dict(title='Kumulierte Einnahme (%)', overlaying='y', side='right', range=[0, 110]),
#    legend=dict(x=0.5, xanchor='center', y=-0.2, orientation='h')
#)

#Einnahmen_Maß.show()

# Visualisierungen

In [165]:
Übersicht.show()
Maße_Tabelle.show()
Einnahmen_Ausgaben.show()
#Einnahmen_Maß_kum.show()
#Einnahmen_Maß.show()

In [ ]:
Übersicht.write_html("Übersicht.html")

In [ ]:
Maße_Tabelle.write_html("Maße_Tabelle.html")

In [ ]:
Einnahmen_Ausgaben.write_html("Einnahmen_Ausgaben.html")

In [167]:
"Interaktives_Kreisdiagramm.html"

'Interaktives_Kreisdiagramm.html'

In [ ]:
#Einnahmen_Maß_kum.write_html("Einnahmen_Maß_kum.html")

In [ ]:
#Einnahmen_Maß.write_html("Einnahmen_Maß.html")